In [1]:
pip install pydicom

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# import os
# import pydicom
# import numpy as np
# import SimpleITK as sitk
# from dicompylercore import dicomparser

# # --- CONFIGURATION ---
# patient_id = '20260104-Copy' # The first failing patient 20250982new
# source_path = '../Data/AMCGH/20260104 - Copy/CT'
# patient_dir = os.path.join(source_path, patient_id)

# print(f"--- DIAGNOSTIC Z-CHECK FOR: {patient_id} ---")

# # 1. Find CT Files & Sort
# ct_files = []
# rt_path = None

# for root, dirs, files in os.walk(patient_dir):
#     for f in files:
#         full = os.path.join(root, f)
#         try:
#             dcm = pydicom.dcmread(full, stop_before_pixels=True, force=True)
#             if dcm.get("Modality") == 'CT':
#                 z = float(dcm.ImagePositionPatient[2])
#                 ct_files.append((z, full))
#             elif dcm.get("Modality") == 'RTSTRUCT':
#                 rt_path = full
#         except: continue

# ct_files.sort(key=lambda x: x[0])
# ct_z_values = [x[0] for x in ct_files]
# ct_paths = [x[1] for x in ct_files]

# print(f"CT Files Found: {len(ct_files)}")
# print(f"CT Z-Range: {min(ct_z_values):.2f} to {max(ct_z_values):.2f}")
# print(f"First 5 CT Zs: {ct_z_values[:5]}")

# if not rt_path:
#     print("CRITICAL: No RTStruct found!")
# else:
#     print(f"RTStruct Found: {os.path.basename(rt_path)}")
    
#     # 2. Check Contours
#     rtss = dicomparser.DicomParser(rt_path)
#     structures = rtss.GetStructures()
    
#     # Find any GTV
#     target_id = None
#     for k, v in structures.items():
#         if 'gtv' in v['name'].lower():
#             target_id = k
#             print(f"Target ROI Found: {v['name']} (ID: {k})")
#             break
            
#     if target_id:
#         contours = rtss.GetStructureCoordinates(target_id)
#         contour_zs = sorted([float(z) for z in contours.keys()])
        
#         print(f"Contour Z-Range: {min(contour_zs):.2f} to {max(contour_zs):.2f}")
#         print(f"First 5 Contour Zs: {contour_zs[:5]}")
        
#         # CHECK OVERLAP
#         print("\n--- MATCHING CHECK ---")
#         matches = 0
#         for cz in contour_zs:
#             # Find closest CT slice
#             diffs = np.abs(np.array(ct_z_values) - cz)
#             min_diff = np.min(diffs)
#             if min_diff < 3.0: # 3mm tolerance
#                 matches += 1
#             else:
#                 if matches == 0: # Print first failure details
#                     print(f"First Mismatch: Contour Z={cz:.4f} -> Closest CT Z={ct_z_values[np.argmin(diffs)]:.4f} (Diff: {min_diff:.4f})")
                    
#         print(f"Total Matches: {matches} / {len(contour_zs)}")
        
#         if matches == 0:
#             print("CONCLUSION: Complete misalignment. Contours are likely on a different frame of reference.")
#         elif matches < len(contour_zs):
#             print("CONCLUSION: Partial match. Some slices missing or misaligned.")
#         else:
#             print("CONCLUSION: Perfect match! Code logic error in previous script.")
            
#     else:
#         print("No GTV found in struct.")

--- DIAGNOSTIC Z-CHECK FOR: 20260104-Copy ---
CT Files Found: 0


ValueError: min() arg is an empty sequence

In [ ]:
# import os
# import numpy as np
# import pandas as pd
# import pydicom
# import SimpleITK as sitk
# from dicompylercore import dicomparser
# from radiomics import featureextractor
# from skimage.draw import polygon
# import logging
# import warnings

# # --- CONFIGURATION ---
# source_name = 'Ahsania'
# source_path = '../Data/AMCGH/20260104'
# output_csv = '../Results/ahsania_radiomics_final.csv'

# # Ensure output directory exists
# os.makedirs('../Results', exist_ok=True)

# # Suppress warnings
# warnings.filterwarnings("ignore")
# logger = logging.getLogger("radiomics")
# logger.setLevel(logging.ERROR)

# # Setup Extractor
# params = {}
# extractor = featureextractor.RadiomicsFeatureExtractor(**params)
# extractor.settings['binWidth'] = 25
# extractor.settings['resampledPixelSpacing'] = [1, 1, 1]
# extractor.settings['interpolator'] = sitk.sitkBSpline
# extractor.enableAllImageTypes()

# print(f"--- STARTING FORCE-MATCH EXTRACTION ({source_name}) ---")

# # --- HELPER 1: Find Files (Recursive & Sorted by Z) ---
# def find_ct_series_and_struct(patient_dir):
#     ct_files = []
#     rt_path = None
#     all_dicoms = []
    
#     for root, dirs, files in os.walk(patient_dir):
#         for f in files:
#             try:
#                 full_path = os.path.join(root, f)
#                 # Force read to ignore extension issues or missing preambles
#                 dcm = pydicom.dcmread(full_path, stop_before_pixels=True, force=True)
#                 mod = dcm.get("Modality", "Unknown")
                
#                 if mod == 'CT':
#                     # Need Z-position for sorting
#                     if 'ImagePositionPatient' in dcm:
#                         z = float(dcm.ImagePositionPatient[2])
#                         all_dicoms.append((z, full_path))
#                 elif mod == 'RTSTRUCT':
#                     rt_path = full_path
#             except: continue
            
#     # Sort CTs by Z-Position (Essential for correct 3D volume)
#     all_dicoms.sort(key=lambda x: x[0])
#     ct_files = [x[1] for x in all_dicoms]
    
#     return ct_files, rt_path

# # --- HELPER 2: Find Tumor ROI ---
# def find_target_roi(structure_dict):
#     # Priority list for tumors
#     target_priorities = ['gtv', 'gtv_t', 'gtv t', 'gtv_total', 'ctv', 'ctv_t', 'ptv']
    
#     # Create a lowercase map for easy searching {CleanName: ROI_ID}
#     clean_map = {}
#     for roi_id, val in structure_dict.items(): # Correct iteration over items()
#         if 'name' in val:
#             clean_name = val['name'].lower().strip().replace(' ','').replace('_','').replace('-','')
#             clean_map[clean_name] = roi_id # Map name to ID

#     # Check priorities
#     for p in target_priorities:
#         clean_p = p.replace(' ','').replace('_','').replace('-','')
        
#         for clean_name, roi_id in clean_map.items():
#             if clean_p in clean_name:
#                 # Return ID and Original Name
#                 return roi_id, structure_dict[roi_id]['name']
                
#     return None, None

# # --- HELPER 3: Force Match Mask Builder ---
# def build_mask_force(ct_files, rtstruct_path, roi_id):
#     # 1. Load CT Series
#     reader = sitk.ImageSeriesReader()
#     reader.SetFileNames(ct_files)
#     try: image_sitk = reader.Execute()
#     except: return None, None, "SITK Load Failed"
    
#     image_arr = sitk.GetArrayFromImage(image_sitk)
#     mask_arr = np.zeros_like(image_arr, dtype=np.uint8)
    
#     # 2. Parse Contours
#     try:
#         rtss = dicomparser.DicomParser(rtstruct_path)
#         contours = rtss.GetStructureCoordinates(roi_id)
#         if not contours: return None, None, "No Contours in ROI"
#     except: return None, None, "Struct Parse Failed"

#     # 3. Build Z-Map (Slice Index -> Physical Z)
#     z_map = {}
#     for i in range(image_sitk.GetDepth()):
#         phys_z = image_sitk.TransformIndexToPhysicalPoint((0,0,i))[2]
#         z_map[i] = phys_z
#     z_values_list = np.array(list(z_map.values()))
    
#     # Get Metadata for Manual Calculation
#     origin = image_sitk.GetOrigin() # (x, y, z)
#     spacing = image_sitk.GetSpacing() # (sx, sy, sz)
    
#     filled_slices = 0
#     min_diff_log = 999.0
    
#     for z_pos_str, slice_data in contours.items():
#         contour_z = float(z_pos_str)
        
#         # Find closest Z-slice
#         diffs = np.abs(z_values_list - contour_z)
#         min_diff = np.min(diffs)
#         if min_diff < min_diff_log: min_diff_log = min_diff
        
#         z_idx = np.argmin(diffs)
        
#         # Tolerance: 5mm (Generous for legacy data)
#         if min_diff > 5.0: continue
        
#         # Normalize Data (Handle nested lists)
#         polygons = []
#         if len(slice_data) > 0:
#             first = slice_data[0]
#             # Check if it is a single contour [x, y, z] or list of contours [[x,y,z], ...]
#             if isinstance(first, list) and len(first) == 3 and isinstance(first[0], float):
#                 polygons.append(slice_data) # Single contour
#             elif isinstance(first, list):
#                 polygons = slice_data # Nested contours
        
#         for points in polygons:
#             r_coords = []
#             c_coords = []
            
#             for p in points:
#                 # MANUAL MAPPING: (World - Origin) / Spacing
#                 try:
#                     pixel_x = (p[0] - origin[0]) / spacing[0]
#                     pixel_y = (p[1] - origin[1]) / spacing[1]
                    
#                     c_coords.append(pixel_x) # Column (X)
#                     r_coords.append(pixel_y) # Row (Y)
#                 except: continue
                
#             if len(r_coords) > 2:
#                 try:
#                     # Draw polygon on the matched Z-slice
#                     rr, cc = polygon(r_coords, c_coords, shape=(image_arr.shape[1], image_arr.shape[2]))
#                     mask_arr[z_idx, rr, cc] = 1
#                     filled_slices += 1
#                 except: continue
                
#     if filled_slices == 0:
#         return None, None, f"Alignment Fail (Min Z-Diff: {min_diff_log:.1f}mm)"
    
#     mask_sitk = sitk.GetImageFromArray(mask_arr)
#     mask_sitk.CopyInformation(image_sitk)
#     return image_sitk, mask_sitk, "Success"

# # --- MAIN LOOP ---
# results = []
# if os.path.exists(source_path):
#     patient_folders = sorted([f for f in os.listdir(source_path) if os.path.isdir(os.path.join(source_path, f))])
#     print(f"Processing {len(patient_folders)} patients...")
    
#     for i, patient_id in enumerate(patient_folders):
#         patient_dir = os.path.join(source_path, patient_id)
        
#         try:
#             # 1. Find Files
#             ct_files, rt_path = find_ct_series_and_struct(patient_dir)
#             if len(ct_files) < 10 or not rt_path:
#                 print(f"[{i+1}] {patient_id}: [SKIP] Files Missing")
#                 continue

#             # 2. Extract
#             rtss = dicomparser.DicomParser(rt_path)
#             roi_id, roi_name = find_target_roi(rtss.GetStructures())
            
#             if roi_id:
#                 image, mask, status = build_mask_force(ct_files, rt_path, roi_id)
#                 if status == "Success":
#                     features = extractor.execute(image, mask)
#                     row = {'PatientID': patient_id, 'Source': source_name, 'ROI_Name': roi_name}
#                     for k, v in features.items():
#                         if 'diagnostics' not in k: row[k] = v
#                     results.append(row)
#                     print(f"[{i+1}] {patient_id}: Success ({roi_name})")
#                 else:
#                     print(f"[{i+1}] {patient_id}: [FAIL] {status}")
#             else:
#                 avail = [s['name'] for k,s in rtss.GetStructures().items()]
#                 print(f"[{i+1}] {patient_id}: [SKIP] No Tumor. Avail: {avail[:3]}")
                
#         except Exception as e:
#             print(f"[{i+1}] {patient_id}: [ERROR] {str(e)}")

#     if results:
#         df = pd.DataFrame(results)
#         df.to_csv(output_csv, index=False)
#         print(f"\nSUCCESS! Saved {len(df)} patients to {output_csv}")
#     else:
#         print("\nNo data extracted.")
# else:
#     print("Source path not found.")

--- STARTING FORCE-MATCH EXTRACTION (Ahsania) ---
Processing 2 patients...
[1] CT: [SKIP] Files Missing
[2] Struct: [SKIP] Files Missing

No data extracted.


In [ ]:
import os
import numpy as np
import pandas as pd
import pydicom
import SimpleITK as sitk
from dicompylercore import dicomparser
from radiomics import featureextractor
from skimage.draw import polygon
import logging
import warnings

# --- CONFIGURATION ---
source_name = 'Ahsania'
# Make sure this path is correct relative to your notebook location
source_path = '../Data/AMCGH/20260104' 
output_csv = '../Results/ahsania_radiomics_final.csv'

# Ensure output directory exists
os.makedirs('../Results', exist_ok=True)

# Suppress warnings
warnings.filterwarnings("ignore")
logger = logging.getLogger("radiomics")
logger.setLevel(logging.ERROR)

# Setup Extractor
params = {}
extractor = featureextractor.RadiomicsFeatureExtractor(**params)
extractor.settings['binWidth'] = 25
extractor.settings['resampledPixelSpacing'] = [1, 1, 1]
extractor.settings['interpolator'] = sitk.sitkBSpline
extractor.enableAllImageTypes()

print(f"--- STARTING FINAL FIXED EXTRACTION ({source_name}) ---")

# --- HELPER 1: Find Files (Force Read + Sort by Z) ---
def find_ct_series_and_struct(patient_dir):
    ct_files = []
    rt_path = None
    all_dicoms = []
    
    # Recursive walk to find all files
    for root, dirs, files in os.walk(patient_dir):
        for f in files:
            try:
                full_path = os.path.join(root, f)
                # Force read to ignore missing headers/extensions
                dcm = pydicom.dcmread(full_path, stop_before_pixels=True, force=True)
                mod = dcm.get("Modality", "Unknown")
                
                if mod == 'CT':
                    # Need Z-position for sorting
                    if 'ImagePositionPatient' in dcm:
                        z = float(dcm.ImagePositionPatient[2])
                        all_dicoms.append((z, full_path))
                elif mod == 'RTSTRUCT':
                    rt_path = full_path
            except: continue
            
    # Sort CTs by Z-Position (Essential for correct 3D volume)
    all_dicoms.sort(key=lambda x: x[0])
    ct_files = [x[1] for x in all_dicoms]
    
    return ct_files, rt_path

# --- HELPER 2: Find Tumor ROI (BUG FIXED) ---
def find_target_roi(structure_dict):
    # Priorities
    target_priorities = ['gtv', 'gtv_t', 'gtv t', 'gtv_total', 'ctv', 'ctv_t', 'ptv', 'ptv_total']
    
    # Create a map of {CleanName: ROI_ID}
    # structure_dict format: {1: {'name': 'GTV', 'color': ...}, 2: ...}
    name_to_id_map = {}
    for roi_id, val in structure_dict.items():
        if isinstance(val, dict) and 'name' in val:
            clean_name = val['name'].lower().strip().replace(' ','').replace('_','').replace('-','')
            name_to_id_map[clean_name] = roi_id

    # Search priorities
    for p in target_priorities:
        clean_p = p.replace(' ','').replace('_','').replace('-','')
        
        for clean_name, roi_id in name_to_id_map.items():
            # Check if priority string is IN the clean name
            if clean_p in clean_name:
                # Return ID and Original Name
                return roi_id, structure_dict[roi_id]['name']
                
    return None, None

# --- HELPER 3: Force Match Mask Builder ---
def build_mask_force(ct_files, rtstruct_path, roi_id):
    # 1. Load CT Series
    reader = sitk.ImageSeriesReader()
    reader.SetFileNames(ct_files)
    try: image_sitk = reader.Execute()
    except: return None, None, "SITK Load Failed"
    
    image_arr = sitk.GetArrayFromImage(image_sitk)
    mask_arr = np.zeros_like(image_arr, dtype=np.uint8)
    
    # 2. Parse Contours
    try:
        rtss = dicomparser.DicomParser(rtstruct_path)
        contours = rtss.GetStructureCoordinates(roi_id)
        if not contours: return None, None, "No Contours in ROI"
    except: return None, None, "Struct Parse Failed"

    # 3. Build Z-Map (Slice Index -> Physical Z)
    z_map = {}
    for i in range(image_sitk.GetDepth()):
        phys_z = image_sitk.TransformIndexToPhysicalPoint((0,0,i))[2]
        z_map[i] = phys_z
    z_values_list = np.array(list(z_map.values()))
    
    # Get Metadata for Manual Calculation
    origin = image_sitk.GetOrigin() # (x, y, z)
    spacing = image_sitk.GetSpacing() # (sx, sy, sz)
    
    filled_slices = 0
    min_diff_log = 999.0
    
    for z_pos_str, slice_data in contours.items():
        contour_z = float(z_pos_str)
        
        # Find closest Z-slice
        diffs = np.abs(z_values_list - contour_z)
        min_diff = np.min(diffs)
        if min_diff < min_diff_log: min_diff_log = min_diff
        
        z_idx = np.argmin(diffs)
        
        # Tolerance: 5mm (Generous for legacy data)
        if min_diff > 5.0: continue
        
        # Normalize Data (Handle nested lists)
        polygons = []
        if len(slice_data) > 0:
            first = slice_data[0]
            # Check if single contour or list of contours
            if isinstance(first, list) and len(first) == 3 and isinstance(first[0], float):
                polygons.append(slice_data) # Single contour
            elif isinstance(first, list):
                polygons = slice_data # Nested
        
        for points in polygons:
            r_coords = []
            c_coords = []
            
            for p in points:
                # MANUAL MAPPING: (World - Origin) / Spacing
                try:
                    pixel_x = (p[0] - origin[0]) / spacing[0]
                    pixel_y = (p[1] - origin[1]) / spacing[1]
                    
                    c_coords.append(pixel_x) # Column
                    r_coords.append(pixel_y) # Row
                except: continue
                
            if len(r_coords) > 2:
                try:
                    rr, cc = polygon(r_coords, c_coords, shape=(image_arr.shape[1], image_arr.shape[2]))
                    mask_arr[z_idx, rr, cc] = 1
                    filled_slices += 1
                except: continue
                
    if filled_slices == 0:
        return None, None, f"Alignment Fail (Min Z-Diff: {min_diff_log:.1f}mm)"
    
    mask_sitk = sitk.GetImageFromArray(mask_arr)
    mask_sitk.CopyInformation(image_sitk)
    return image_sitk, mask_sitk, "Success"

# --- MAIN LOOP ---
results = []
if os.path.exists(source_path):
    patient_folders = sorted([f for f in os.listdir(source_path) if os.path.isdir(os.path.join(source_path, f))])
    print(f"Processing {len(patient_folders)} patients...")
    
    for i, patient_id in enumerate(patient_folders):
        patient_dir = os.path.join(source_path, patient_id)
        
        try:
            # 1. Find Files
            ct_files, rt_path = find_ct_series_and_struct(patient_dir)
            if len(ct_files) < 10 or not rt_path:
                print(f"[{i+1}] {patient_id}: [SKIP] Files Missing")
                continue

            # 2. Extract
            rtss = dicomparser.DicomParser(rt_path)
            roi_id, roi_name = find_target_roi(rtss.GetStructures())
            
            if roi_id:
                image, mask, status = build_mask_force(ct_files, rt_path, roi_id)
                if status == "Success":
                    features = extractor.execute(image, mask)
                    row = {'PatientID': patient_id, 'Source': source_name, 'ROI_Name': roi_name}
                    for k, v in features.items():
                        if 'diagnostics' not in k: row[k] = v
                    results.append(row)
                    print(f"[{i+1}] {patient_id}: Success ({roi_name})")
                else:
                    print(f"[{i+1}] {patient_id}: [FAIL] {status}")
            else:
                avail = [s['name'] for k,s in rtss.GetStructures().items()]
                print(f"[{i+1}] {patient_id}: [SKIP] No Tumor. Avail: {avail[:3]}")
                
        except Exception as e:
            print(f"[{i+1}] {patient_id}: [ERROR] {str(e)}")

    if results:
        df = pd.DataFrame(results)
        df.to_csv(output_csv, index=False)
        print(f"\nSUCCESS! Saved {len(df)} patients to {output_csv}")
    else:
        print("\nNo data extracted.")
else:
    print("Source path not found.")

--- STARTING FINAL FIXED EXTRACTION (Ahsania) ---
Source path not found.


In [9]:
import os
import pandas as pd
import SimpleITK as sitk
from rt_utils import RTStructBuilder
from radiomics import featureextractor
import logging
import warnings
import pydicom

# Suppress warnings
warnings.filterwarnings("ignore")
logger = logging.getLogger("radiomics")
logger.setLevel(logging.ERROR)

# --- CONFIGURATION ---
source_name = 'Ahsania'
source_path = '../Data/AMCGH/20260104'
output_csv = '../Results/ahsania_radiomics_final.csv'

# Setup Extractor
params = {}
extractor = featureextractor.RadiomicsFeatureExtractor(**params)
extractor.settings['binWidth'] = 25
extractor.settings['resampledPixelSpacing'] = [1, 1, 1]
extractor.settings['interpolator'] = sitk.sitkBSpline
extractor.enableAllImageTypes()

print(f"--- STARTING AHSANIA RADIOMICS (FINAL MODE) ---")

# --- HELPER: Find Tumor ---
def find_target_roi(rtstruct):
    try:
        rois = rtstruct.get_roi_names()
    except: return None
    
    # Ahsania Priorities
    target_priorities = ['ptv', 'ptv_total', 'ptv total', 'gtv', 'gtv_t']
    
    avail_clean = {name.lower().strip().replace(' ','').replace('_','').replace('-',''): name for name in rois}
    
    for priority in target_priorities:
        clean_p = priority.replace(' ','').replace('_','').replace('-','')
        for clean_a, original_name in avail_clean.items():
            if clean_p in clean_a:
                return original_name
    return None

# --- MAIN LOOP ---
results = []
patient_folders = sorted([f for f in os.listdir(source_path) if os.path.isdir(os.path.join(source_path, f))])

for i, patient_id in enumerate(patient_folders):
    patient_dir = os.path.join(source_path, patient_id)
    ct_folder = os.path.join(patient_dir, 'CT')
    struct_folder = os.path.join(patient_dir, 'Struct')
    
    # Check if cleaning happened
    if not os.path.exists(ct_folder) or not os.path.exists(struct_folder):
        print(f"[{i+1}] {patient_id}: [SKIP] Clean folders not found. Run Cleaner script!")
        continue
        
    # Find RTStruct file inside the new /Struct folder
    rt_path = None
    if len(os.listdir(struct_folder)) > 0:
        rt_path = os.path.join(struct_folder, os.listdir(struct_folder)[0])
            
    if not rt_path:
        print(f"[{i+1}] {patient_id}: [SKIP] No Struct file found")
        continue

    try:
        # 1. Build RTStruct
        # Now we point specifically to the clean folders
        rtstruct = RTStructBuilder.create_from(
            dicom_series_path=ct_folder, 
            rt_struct_path=rt_path
        )
        
        # 2. Find ROI
        roi_name = find_target_roi(rtstruct)
        
        if roi_name:
            # 3. Mask & Extract
            mask_3d = rtstruct.get_roi_mask_by_name(roi_name)
            
            # Read image using SITK
            reader = sitk.ImageSeriesReader()
            dicom_names = reader.GetGDCMSeriesFileNames(ct_folder)
            reader.SetFileNames(dicom_names)
            image_sitk = reader.Execute()
            
            # Convert mask to SITK
            mask_sitk = sitk.GetImageFromArray(mask_3d.astype(int).transpose(2, 0, 1))
            mask_sitk.CopyInformation(image_sitk)
            
            features = extractor.execute(image_sitk, mask_sitk)
            
            row = {'PatientID': patient_id, 'Source': source_name, 'ROI_Name': roi_name}
            for k, v in features.items():
                if 'diagnostics' not in k: row[k] = v
            results.append(row)
            print(f"[{i+1}] {patient_id}: Success ({roi_name})")
        else:
            print(f"[{i+1}] {patient_id}: [SKIP] No Tumor Found")
            
    except Exception as e:
        print(f"[{i+1}] {patient_id}: [ERROR] {str(e)}")

# Save
if results:
    df = pd.DataFrame(results)
    df.to_csv(output_csv, index=False)
    print(f"\nSUCCESS! Saved {len(df)} patients.")
else:
    print("\nNo data extracted.")

--- STARTING AHSANIA RADIOMICS (FINAL MODE) ---
[1] CT: [SKIP] Clean folders not found. Run Cleaner script!
[2] Struct: [SKIP] Clean folders not found. Run Cleaner script!

No data extracted.
